In [1]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [14]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
load_dotenv(override=True)
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
## imports 
import torch 
import json 
from datasets import load_dataset 
from transformers import AutoTokenizer, AutoModelForCausalLM 
from trl import SFTTrainer, SFTConfig 
from peft import LoraConfig, PeftModel 
from transformers.utils import get_json_schema 

In [16]:
MODEL = "google/functiongemma-270m-it" 
OUTPUT_DIR = "functiongemma-270m-ft" 
MERGED_OUTPUT_DIR = "merged_model"
DATASET_PATH = "dataset.jsonl"

In [17]:
# Manual tool definations 

def control_light(action: str, device_name: str = None, brightness: int = None, color: str = None) -> str:
    """
    Control smart lights - turn on, off, dim, or change color.
    
    Args:
        action: Action to perform: on, off, dim, toggle
        device_name: Name of the light or room
        brightness: Brightness level 0-100
        color: Color name or hex code
    """
    return "result"

def set_timer(duration: str, label: str = None) -> str:
    """
    Set a countdown timer.
    
    Args:
        duration: Duration like '5 minutes' or '1 hour'
        label: Optional label for the timer
    """
    return "result"

def set_alarm(time: str, label: str = None) -> str:
    """
    Set an alarm for a specific time.
    
    Args:
        time: Time for alarm like '7am' or '14:30'
        label: Optional label
    """
    return "result"

def create_calendar_event(title: str, date: str = None, time: str = None, duration: int = None) -> str:
    """
    Create a calendar event.
    
    Args:
        title: Event title
        date: Date like 'tomorrow' or '2024-01-15'
        time: Time like '3pm'
        duration: Duration in minutes
    """
    return "result"

def add_task(text: str, priority: str = None) -> str:
    """
    Add a task to the to-do list.
    
    Args:
        text: Task description
        priority: Priority level
    """
    return "result"

def web_search(query: str) -> str:
    """
    Search the web for information.
    
    Args:
        query: Search query
    """
    return "result"

def get_system_info() -> str:
    """
    Get current system state including timers, calendar, tasks, devices, and weather.
    """
    return "result"

def thinking(prompt: str) -> str:
    """
    Use for complex queries requiring reasoning, math, coding, or multi-step analysis.
    
    Args:
        prompt: The user's original prompt
    """
    return "result"

def nonthinking(prompt: str) -> str:
    """
    Use for simple queries, greetings, factual questions not requiring deep reasoning.
    
    Args:
        prompt: The user's original prompt
    """
    return "result"


In [18]:
# Generate tool Schemas
TOOLS = [
    get_json_schema(control_light),
    get_json_schema(set_timer),
    get_json_schema(set_alarm),
    get_json_schema(create_calendar_event),
    get_json_schema(add_task),
    get_json_schema(web_search),
    get_json_schema(get_system_info),
    get_json_schema(thinking),
    get_json_schema(nonthinking)
]

In [19]:
TOOLS[0]

{'type': 'function',
 'function': {'name': 'control_light',
  'description': 'Control smart lights - turn on, off, dim, or change color.',
  'parameters': {'type': 'object',
   'properties': {'action': {'type': 'string',
     'description': 'Action to perform: on, off, dim, toggle'},
    'device_name': {'type': 'string',
     'description': 'Name of the light or room'},
    'brightness': {'type': 'integer', 'description': 'Brightness level 0-100'},
    'color': {'type': 'string', 'description': 'Color name or hex code'}},
   'required': ['action']},
  'return': {'type': 'string'}}}

In [20]:
SYSTEM_PROMPT = "You are a model that can do function calling with the following functions"

In [24]:
def rebuild_with_proper_schema(sample):
    '''Rebuild sample with proper formatted tool schema'''
    messages = sample['messages'] 

    # Find the tool call 
    tool_name = None
    tool_args = None 
    user_content = None 

    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant' and 'tool_calls' in msg:
            tc = msg['tool_calls'][0]['function']
            tool_name = tc['name']
            tool_args = tc['arguments']
    
    if not all([user_content, tool_name]):
        return sample 
    
    if tool_args is None:
        tool_args = {} 
    
    return {
        "messages": [
            {"role": "developer", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "tool_calls": [{"type": "function", "function": {"name": tool_name, "arguments": tool_args}}]}
        ],
        "tools": TOOLS
    }

In [ ]:
def train():
    print("Loading Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)

    print(f"Loading dataset from {DATASET_PATH}...")
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train") 
    print(f"Dataset size: {len(dataset)}")

    print("Rebuilding with proper tool schemas...")
    dataset = dataset.map(rebuild_with_proper_schema, remove_columns=dataset.column_names) 

    # ====== FIX 1: Train/Validation Split to detect overfitting ======
    print("\nSplitting dataset into train/validation (90/10)...")
    split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
    print(f"Train samples: {len(train_dataset)}, Validation samples: {len(eval_dataset)}")

    print("\n--- Sample dataset entry ---")
    sample = train_dataset[0]
    print(f"User: {sample['messages'][1]['content']}")
    print(f"Function: {sample['messages'][2]['tool_calls'][0]['function']['name']}")
    print(f"Arguments: {sample['messages'][2]['tool_calls'][0]['function']['arguments']}")

    debug_msg = tokenizer.apply_chat_template( 
        sample['messages'],
        tools=sample['tools'],
        add_generation_prompt=False,
        tokenize=False 
    )
    print(f"\n--- Tokenized length: {len(tokenizer.encode(debug_msg))} tokens ---\n")

    print("Loading model...") 
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        torch_dtype=torch.bfloat16, 
        device_map="auto",
        attn_implementation="eager"
    )
    
    print(f"Device: {model.device}")
    print(f"DType: {model.dtype}")

    # ====== FIX 2: Increased LoRA dropout for regularization ======
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,  # Increased from 0.05 to 0.1 for better regularization
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM"
    )

    # ====== FIX 3: Adjusted training config with evaluation ======
    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        max_length=1024,
        packing=False,
        num_train_epochs=2,  # Reduced from 4 to 2 epochs
        per_device_train_batch_size=1, 
        per_device_eval_batch_size=1,  # Added eval batch size
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        optim="paged_adamw_32bit",
        logging_steps=5,
        save_strategy="steps",  # Changed to steps for early stopping
        save_steps=50,  # Save every 50 steps
        eval_strategy="steps",  # Enable evaluation during training
        eval_steps=50,  # Evaluate every 50 steps
        learning_rate=2e-5,
        bf16=True,
        lr_scheduler_type="cosine",
        report_to="none",
        load_best_model_at_end=True,  # Load best model when training ends
        metric_for_best_model="eval_loss",  # Use eval loss to determine best model
        greater_is_better=False,  # Lower eval loss is better
        save_total_limit=3,  # Keep only 3 best checkpoints
    )

    # ====== FIX 4: Early stopping callback ======
    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=3,  # Stop if no improvement for 3 evaluations
        early_stopping_threshold=0.01  # Minimum improvement threshold
    )

    # Create Trainer with eval dataset and early stopping
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,  # Now using split train set
        eval_dataset=eval_dataset,  # Added validation set
        args=args,
        peft_config=peft_config,
        processing_class=tokenizer,
        callbacks=[early_stopping],  # Added early stopping
    )

    print(f"Starting training on {len(train_dataset)} samples...")
    print(f"Evaluating on {len(eval_dataset)} samples...")
    print(f"Functions: control_light, set_timer, set_alarm, create_calendar_event,")
    print(f"           add_task, web_search, get_system_info, thinking, nonthinking")
    print("=" * 60)
    print("Early stopping enabled: will stop if eval_loss doesn't improve for 3 evals")
    print("=" * 60)

    trainer.train()

    print("Saving adapter ...")
    trainer.save_model(OUTPUT_DIR)


    # Free memory
    del model
    del trainer
    torch.cuda.empty_cache()

    # Merge
    print("\nMerging adapter into base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    merged_model = merged_model.merge_and_unload()
    
    print(f"Saving merged model to: {MERGED_OUTPUT_DIR}")
    merged_model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

    print("\n" + "=" * 60)
    print("TRAINING COMPLETE!")
    print("=" * 60)
    print(f"Merged model saved to: {MERGED_OUTPUT_DIR}")
    print("\nTo test the model, run:")
    print("  python -m core.router")

In [30]:
train()

Loading Tokenizer...
Loading dataset from dataset.jsonl...
Dataset size: 913
Rebuilding with proper tool schemas...

--- Sample dataset entry ---
User: Add new task: fix leaky faucet
Function: add_task
Arguments: {'prompt': None, 'title': None, 'date': None, 'time': None, 'duration': None, 'label': None, 'action': None, 'device_name': None, 'query': None, 'text': 'fix leaky faucet', 'brightness': None, 'color': None}

--- Tokenized length: 2000 tokens ---

Loading model...


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 445.98it/s, Materializing param=model.norm.weight]                                


Device: cuda:0
DType: torch.bfloat16


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Starting training on 913 samples...
Functions: control_light, set_timer, set_alarm, create_calendar_event,
           add_task, web_search, get_system_info, thinking, nonthinking


...

Saving adapter ...

Merging adapter into base model...


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 408.82it/s, Materializing param=model.norm.weight]                                


Saving merged model to: merged_model


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]



TRAINING COMPLETE!
Merged model saved to: merged_model

To test the model, run:
  python -m core.router
